In [21]:
import pandas as pd

from io import BytesIO
import requests
from PIL import Image
from qdrant_client import QdrantClient
from qdrant_client.http import models
import torch
from transformers import CLIPProcessor, CLIPModel

In [18]:
# Function to download image and generate embedding

def generate_image_embedding(image_url: str, processor: CLIPProcessor, model: CLIPModel):
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))
    inputs_image = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        image_features = model.get_image_features(**inputs_image)
    return image_features / image_features.norm(p=2, dim=-1, keepdim=True)

In [2]:
# read modalab data
modalab_df = pd.read_csv(
    '../data/products.csv'
)

In [ ]:
# Load CLIP Model

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [ ]:
# Initialize Qdrant vector db
client = QdrantClient(
    url='XXX',
    port=6333,
    https=True,
    api_key='XXX'
)

In [12]:
client.recreate_collection(
    collection_name="modalab_products",
    vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE)
)

/var/folders/yx/lzqg5b6n7hzg_qtlrbylts0m0000gn/T/ipykernel_15021/4096197671.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [ ]:
# Store embeddings and metadata in Qdrant
for _, item in modalab_df.iterrows():
    if not item['main_image']:
        continue

    try:
        embedding = generate_image_embedding(
            item["main_image"],
            processor,
            model
        ).numpy().flatten()  # Convert embedding to 1D array

        client.upsert(
            collection_name="modalab_products",
            points=[
                models.PointStruct(
                    id=item["id"], 
                    vector=embedding.tolist(),  # Convert to list
                    payload=item.to_dict()
                )
            ]
        )
    except:
        print(f"Couldn't index image: {item['main_image']}")

Couldn't index image: https://storage.googleapis.com/images_modalab/designers/Emuna%20/Evolution%20Pant%20Set/serenity_evolution-pant-set-black_product.webp 
Couldn't index image: https://storage.googleapis.com/images_modalab/designers/Emuna%20/Breathe%20Pant%20Set/serenity_breathe-pant-set-gray_product.webp 
